In [ ]:
suppressPackageStartupMessages({
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
})

In [ ]:
res_df = readRDS("../data/res_gene_corr_df_intra_inter_joined.RDS")

In [ ]:
lineages_to_omit <- c("GABAergic", "Glutamatergic")

stouffer_df <- res_df %>%
    filter(!trajectory_refined %in% lineages_to_omit) %>%
    mutate(Spearman_pvalue = ifelse(Spearman_pvalue == 0, 1e-300, Spearman_pvalue),
           Z_score = sign(Spearman) * qnorm(Spearman_pvalue / 2, lower.tail = FALSE)) %>%

    group_by(gene_ID, gene_short_name) %>%
    summarize(
        mean_Spearman = mean(Spearman, na.rm = TRUE),
        median_Spearman = median(Spearman, na.rm = TRUE),
        k_trajectories = n(),                           
        sum_Z = sum(Z_score, na.rm = TRUE),     
        n_positive_0.2 = sum(Spearman > 0.2, na.rm = TRUE), 
        n_positive_0.3 = sum(Spearman > 0.3, na.rm = TRUE), 
        n_positive_0.4 = sum(Spearman > 0.4, na.rm = TRUE), 
        n_negative = sum(Spearman < -0.3, na.rm = TRUE),
        Spearman = first(global_cor),
        .groups = "drop"
    ) %>%

    mutate(
        Stouffer_Z = sum_Z / sqrt(k_trajectories),      
        Meta_Pvalue = 2 * pnorm(-abs(Stouffer_Z)),      
        Meta_FDR = p.adjust(Meta_Pvalue, method = "BH") 
    ) %>%

    arrange(desc(Stouffer_Z))

In [ ]:
stouffer_Z_5_median_0.7_0.4_n_6 = stouffer_df %>%
    filter(Stouffer_Z > 5 & Meta_FDR < 0.01 & n_negative < 2 & n_positive_0.3 > 5) %>%
    filter(round(median_Spearman, 2) >= 0.4) %>%
    filter(Spearman > 0.7)

stouffer_Z_5_median_0.7_0.4_n_6 %>% as.data.frame() %>% print()

In [ ]:
openxlsx::write.xlsx(stouffer_Z_5_median_0.7_0.4_n_6, "s_nucleolar_genes_0.7_0.4_n_6_41.xlsx")

In [ ]:
negative_genes = stouffer_df %>%
    filter(Stouffer_Z < -5 & n_negative > 5 & n_positive_0.3 < 2 & median_Spearman < -0.4) %>%
    arrange(Stouffer_Z)
negative_genes

In [ ]:
openxlsx::write.xlsx(negative_genes, "s_nucleolar_genes_neg_0.4_n_6_20.xlsx")